In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FUENTE = RAIZ / "data" / "processed" / "seguimiento_academico_semana3_preparado.csv"
SALIDA = RAIZ / "outputs"
PROCESADOS = RAIZ / "data" / "processed"
SALIDA.mkdir(parents=True, exist_ok=True)
PROCESADOS.mkdir(parents=True, exist_ok=True)
assert FUENTE.is_file(), f"No se encontró la fuente: {FUENTE}"

origen = pd.read_csv(FUENTE, dtype={"documento": "string", "ficha": "string"},
                     encoding="utf-8-sig")
df = origen.copy()
for campo in ["nota_python", "nota_datos", "asistencia"]:
    df[campo] = pd.to_numeric(df[campo], errors="coerce")
for campo in ["fecha_actualizacion", "fecha_corte"]:
    df[campo] = pd.to_datetime(df[campo], errors="coerce")

sha256 = hashlib.sha256(FUENTE.read_bytes()).hexdigest()
print("Fuente:", FUENTE.name, "| filas:", len(df))
print("Huella SHA-256:", sha256[:16], "...")
print(df[["documento", "nota_python", "nota_datos", "asistencia"]].head(3))

df["conflicto_identidad"] = (
    df["documento"].notna() & df["documento"].duplicated(keep=False)
)
df["fecha_fuera_corte"] = (
    df["fecha_actualizacion"] > df["fecha_corte"]
).fillna(False)

print("Filas:", len(df))
print("Registros en conflicto:", int(df["conflicto_identidad"].sum()))
print("Actualizaciones posteriores al corte:", int(df["fecha_fuera_corte"].sum()))

notas = ["nota_python", "nota_datos"]
df["numero_notas"] = df[notas].notna().sum(axis=1)
df["promedio_dos_notas"] = (
    (df["nota_python"] + df["nota_datos"]) / 2
).where(df[notas].notna().all(axis=1))

UMBRAL_ASISTENCIA = 80
UMBRAL_PROMEDIO = 3.5
df["brecha_asistencia_pp"] = df["asistencia"] - UMBRAL_ASISTENCIA

print(df[["documento", "numero_notas", "promedio_dos_notas", "brecha_asistencia_pp"]].head())

df["programa_limpio"] = (
    df["programa"].astype("string").str.strip().str.casefold()
    .replace({
        "adso": "ADSO",
        "ciencia datos python": "Ciencia de datos con Python",
        "ciencia de datos con python": "Ciencia de datos con Python",
    })
)
df["ciudad_limpia"] = (
    df["ciudad"].astype("string").str.strip().str.casefold()
    .replace({"medellín": "Medellín", "medellin": "Medellín",
              "bello": "Bello", "itagüí": "Itagüí", "itagui": "Itagüí",
              "envigado": "Envigado"})
)
print(df[["ciudad", "ciudad_limpia"]].tail(9))

codigo_ciudad = {"Medellín": "MED", "Bello": "BEL",
                 "Itagüí": "ITA", "Envigado": "ENV"}
df["codigo_ciudad"] = df["ciudad_limpia"].map(codigo_ciudad)
df["ciudad_por_revisar"] = (
    df["ciudad"].notna() & df["codigo_ciudad"].isna()
)

print(df[["ciudad", "ciudad_limpia", "codigo_ciudad"]].tail(9))
print("Categorías sin equivalencia:", df.loc[df["ciudad_por_revisar"], "ciudad"].unique())

df["notas_completas"] = np.where(df["numero_notas"].eq(2), "Sí", "No")

df["estado_asistencia"] = np.select(
    [df["asistencia"].isna(), df["asistencia"].lt(UMBRAL_ASISTENCIA)],
    ["Sin dato", "Bajo umbral didáctico"],
    default="En/sobre umbral didáctico"
)

sin_fecha = df["fecha_actualizacion"].isna() | df["fecha_corte"].isna()
incompleto = (df["promedio_dos_notas"].isna() | df["asistencia"].isna() | df["documento"].isna())

df["prioridad_revision"] = np.select(
    [
        df["conflicto_identidad"],
        df["fecha_fuera_corte"],
        sin_fecha,
        incompleto,
        (df["promedio_dos_notas"].lt(UMBRAL_PROMEDIO) & df["asistencia"].lt(UMBRAL_ASISTENCIA)),
        (df["promedio_dos_notas"].lt(UMBRAL_PROMEDIO) | df["asistencia"].lt(UMBRAL_ASISTENCIA)),
    ],
    ["Revisar identidad", "Revisar fecha de corte", "Fecha no verificable",
     "Datos insuficientes", "Prioridad alta", "Prioridad preventiva"],
    default="Seguimiento habitual"
)
print(df["prioridad_revision"].value_counts(dropna=False))

assert len(df) == len(origen)
assert df["promedio_dos_notas"].dropna().between(0, 5).all()
assert df.loc[df["numero_notas"].lt(2), "promedio_dos_notas"].isna().all()

base_analizable = df.loc[
    ~df["conflicto_identidad"] & ~df["fecha_fuera_corte"] &
    df["fecha_actualizacion"].notna() &
    df["documento"].notna() & df["promedio_dos_notas"].notna() &
    df["asistencia"].notna()
].copy()

indicadores = pd.DataFrame([
    {"indicador": "Registros preparados", "valor": len(df), "universo": "Todas las filas recibidas"},
    {"indicador": "Registros con identidad en conflicto", "valor": int(df["conflicto_identidad"].sum()), "universo": "Todas las filas recibidas"},
    {"indicador": "Registros aptos para el análisis", "valor": len(base_analizable), "universo": "Sin conflicto, fecha válida y variables completas"},
    {"indicador": "Promedio de dos notas (base apta)", "valor": round(base_analizable["promedio_dos_notas"].mean(), 2), "universo": "Solo registros aptos"},
])
print(indicadores.to_string(index=False))

bitacora = pd.DataFrame([
    ["T01", "nota_python + nota_datos", "promedio_dos_notas", "Promediar solo si ambas notas existen", "Sin dato si falta una"],
    ["T02", "programa, ciudad", "programa_limpio, ciudad_limpia", "Normalizar texto con equivalencias", "No inferir 'BELL0'"],
    ["T03", "documento, fecha, notas, asistencia", "prioridad_revision", "np.select con precedencia", "No clasifica personas"],
], columns=["id_regla", "campos_origen", "campos_destino", "regla", "limitacion"])

ruta_df = PROCESADOS / "semana5_seguimiento_transformado.csv"
ruta_ind = SALIDA / "semana5_indicadores.csv"
ruta_reglas = SALIDA / "semana5_bitacora_transformaciones.csv"

df.to_csv(ruta_df, index=False, encoding="utf-8-sig")
indicadores.to_csv(ruta_ind, index=False, encoding="utf-8-sig")
bitacora.to_csv(ruta_reglas, index=False, encoding="utf-8-sig")

verificado = pd.read_csv(ruta_df, dtype={"documento": "string"})
assert len(verificado) == len(df)
print("¡Exportación exitosa! Archivos generados:")
print(f" - {ruta_df.name}")
print(f" - {ruta_ind.name}")
print(f" - {ruta_reglas.name}")